# v2 Safety Analysis -- A2A-CMA

This notebook is the canonical analysis pipeline for the v2 paper. Given a directory of saved episode JSONs (the shape that `Conversation.save_conversation` writes), it produces:

* the **scenario x anomaly heatmap** (Figure 1 candidate),
* anomaly incidence broken down by seller archetype,
* a reward summary by archetype using the v3 anomaly-typed   reward (`rl.policy.reward_from_anomalies`), and
* a regret-rate bar chart per archetype when the v2.5 oracle   judge has been run (`evaluation.judge_episode`).

It runs out-of-the-box on synthetic episodes generated by `tools.generate_synthetic_episodes`, and on real episodes once you point `RUN_DIR` at the directory written by the A2A-CMA CLI.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from MarkAnomaly import PostDataProcessor
from rl.policy import reward_from_anomalies, DEFAULT_REWARD_COEFFS

# Directory of saved episode JSONs. Override to analyse real runs:
#     RUN_DIR = 'runs/real'
RUN_DIR = 'runs/synthetic'


In [ ]:
# If RUN_DIR is empty (or missing) bootstrap with synthetic data so the
# notebook is runnable on a fresh checkout. The synthetic generator is
# stdlib-only and deterministic given the seed.
run_dir = Path(RUN_DIR)
needs_bootstrap = (
    not run_dir.exists()
    or not any(run_dir.rglob('*.json'))
)
if needs_bootstrap:
    from tools.generate_synthetic_episodes import generate
    generate(RUN_DIR, n_per_scenario=5, seed=0)
    print(f'Bootstrapped synthetic episodes into {RUN_DIR!r}.')
else:
    print(f'Using existing episodes in {RUN_DIR!r}.')


In [ ]:
# Load every JSON under RUN_DIR, compute anomalies + reward, and flatten
# into a single DataFrame (one row per episode).
episode_paths = sorted(Path(RUN_DIR).rglob('*.json'))
processor = PostDataProcessor()

records = []
for path in episode_paths:
    with path.open() as f:
        ep = json.load(f)
    anomalies = processor.calculate_anomalies(ep)
    scenario_id = (ep.get('scenario') or {}).get('scenario_id', 'unknown')
    archetype = ep.get('seller_archetype') or (ep.get('scenario') or {}).get('seller_archetype', 'unknown')
    record = {
        'scenario_id': scenario_id,
        'seller_archetype': archetype,
        'negotiation_result': ep.get('negotiation_result'),
        'regret_verdict': ep.get('regret_verdict'),
        'reward': reward_from_anomalies(anomalies, ep),
    }
    record.update(anomalies)
    records.append(record)

df = pd.DataFrame.from_records(records)
print(f'Loaded {len(df)} episodes across {df["scenario_id"].nunique()} scenarios '
      f'and {df["seller_archetype"].nunique()} archetypes.')
df.head()


## Figure 1 -- scenario x anomaly heatmap

Rows are scenarios, columns are the boolean anomaly fields produced by `MarkAnomaly.PostDataProcessor.calculate_anomalies`. Each cell is the fraction of episodes for that scenario that fired the anomaly. Saved to `data_postprocess/figures/v2_anomaly_heatmap.png`.

In [ ]:
BOOL_ANOMALY_COLS = [
    c for c in df.columns
    if c not in ('scenario_id', 'seller_archetype', 'negotiation_result',
                 'regret_verdict', 'reward')
    and df[c].dtype == bool
]

heatmap = (
    df.groupby('scenario_id')[BOOL_ANOMALY_COLS].mean().sort_index()
)

fig, ax = plt.subplots(figsize=(max(8, 0.7 * len(BOOL_ANOMALY_COLS)),
                                max(4, 0.5 * len(heatmap))))
im = ax.imshow(heatmap.values, aspect='auto', cmap='Reds', vmin=0.0, vmax=1.0)
ax.set_xticks(range(len(heatmap.columns)))
ax.set_xticklabels(heatmap.columns, rotation=45, ha='right')
ax.set_yticks(range(len(heatmap.index)))
ax.set_yticklabels(heatmap.index)
ax.set_title('Figure 1 -- Scenario x Anomaly Incidence (A2A-CMA v2)')
for i in range(heatmap.shape[0]):
    for j in range(heatmap.shape[1]):
        val = heatmap.values[i, j]
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                color='white' if val > 0.5 else 'black', fontsize=8)
fig.colorbar(im, ax=ax, label='incidence rate')
fig.tight_layout()

fig_dir = Path('data_postprocess/figures')
fig_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_dir / 'v2_anomaly_heatmap.png', dpi=150, bbox_inches='tight')
print(f'Saved heatmap to {fig_dir / "v2_anomaly_heatmap.png"!s}.')
plt.show()


## Anomaly incidence by archetype

Each cell is the fraction of episodes (within a given seller archetype) where the anomaly fired. Saved to `data_postprocess/tables/anomaly_by_archetype.csv`.

In [ ]:
by_archetype = df.groupby('seller_archetype')[BOOL_ANOMALY_COLS].mean().sort_index()

tables_dir = Path('data_postprocess/tables')
tables_dir.mkdir(parents=True, exist_ok=True)
by_archetype.to_csv(tables_dir / 'anomaly_by_archetype.csv')
print(f'Saved table to {tables_dir / "anomaly_by_archetype.csv"!s}.')

by_archetype.style.background_gradient(cmap='Reds', axis=None).format('{:.2f}')


## Reward summary by archetype

Mean / std / min / max of the v3 anomaly-typed reward per archetype. Coefficients live in `rl.policy.DEFAULT_REWARD_COEFFS`. Saved to `data_postprocess/tables/reward_by_archetype.csv`.

In [ ]:
reward_summary = df.groupby('seller_archetype')['reward'].agg(['mean', 'std', 'min', 'max']).sort_index()
reward_summary.to_csv(tables_dir / 'reward_by_archetype.csv')
print(f'Saved table to {tables_dir / "reward_by_archetype.csv"!s}.')
print('\nReward coefficients in use:')
for k, v in DEFAULT_REWARD_COEFFS.items():
    print(f'  {k:<42s} {v:+.2f}')
reward_summary


## Regret rate by archetype (when judge available)

Bar chart of the v2.5 oracle-judge regret rate per archetype. This cell is a no-op (with a printed note) when no episodes carry a `regret_verdict` field, which is the case for fresh synthetic data.

In [ ]:
from evaluation import RegretVerdict, aggregate_regret_rate

judged = df[df['regret_verdict'].notna() & (df['regret_verdict'] != '')]
if judged.empty:
    print('No episodes have a regret_verdict attached; skipping regret-rate plot.')
    print('Run `evaluation.judge_episode(ep)` over RUN_DIR to populate the field.')
else:
    rates = {}
    for arch, sub in judged.groupby('seller_archetype'):
        verdicts = []
        for v in sub['regret_verdict']:
            try:
                verdicts.append(RegretVerdict(v))
            except ValueError:
                continue
        rates[arch] = aggregate_regret_rate(verdicts)['regret_rate']
    fig, ax = plt.subplots(figsize=(8, 4))
    archetypes = sorted(rates)
    ax.bar(archetypes, [rates[a] for a in archetypes], color='C3')
    ax.set_ylim(0.0, 1.0)
    ax.set_ylabel('regret_rate')
    ax.set_title('User-regret rate by seller archetype (v2.5 oracle judge)')
    ax.set_xticklabels(archetypes, rotation=30, ha='right')
    fig.tight_layout()
    fig.savefig(fig_dir / 'v2_regret_rate.png', dpi=150, bbox_inches='tight')
    plt.show()


## How to regenerate this analysis on real data

```bash
python -m a2a_cma_cli run-all --output runs/real ...
```

Then in this notebook set `RUN_DIR = 'runs/real'` and re-run all cells.